# Comparação de classificadores de crédito

O notebook compara regressão logística, Naive Bayes, random forest e SVM. O SVM é ajustado por validação cruzada no treino. O teste fica reservado para a comparação final.

> `pickle` pode executar código. Este notebook aceita apenas o artefato versionado neste repositório privado; não substitua o arquivo por um download não verificado.

In [1]:
import pickle
import warnings
from pathlib import Path

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

In [2]:
candidates = [
    Path('dados/credit.pkl'),
    Path('classificadores-de-credito/dados/credit.pkl'),
]
legacy_path = next((path for path in candidates if path.is_file()), None)
if legacy_path is None:
    raise FileNotFoundError('credit.pkl não encontrado')
warnings.warn('Abrindo o pickle legado incluído no repositório.', UserWarning)
with legacy_path.open('rb') as arquivo:
    X_train, y_train, X_test, y_test = pickle.load(arquivo)[:4]
len(X_train), len(X_test)

C:\Users\AMD\AppData\Local\Temp\ipykernel_23700\603676219.py:8: UserWarning: Abrindo o pickle legado incluído no repositório.
  warnings.warn('Abrindo o pickle legado incluído no repositório.', UserWarning)


(1500, 500)

In [3]:
models = {
    'regressao_logistica': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(random_state=42)),
    ]),
    'naive_bayes': Pipeline([
        ('scaler', StandardScaler()),
        ('model', GaussianNB()),
    ]),
    'random_forest': RandomForestClassifier(
        n_estimators=100, class_weight='balanced', random_state=42
    ),
}
rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    rows.append({
        'modelo': name,
        'acuracia': accuracy_score(y_test, predictions),
        'macro_f1': f1_score(y_test, predictions, average='macro', zero_division=0),
        'balanced_accuracy': balanced_accuracy_score(y_test, predictions),
    })

In [4]:
svm = GridSearchCV(
    Pipeline([('scaler', StandardScaler()), ('model', SVC())]),
    {'model__kernel': ['linear', 'rbf', 'poly'], 'model__C': [0.1, 1, 10]},
    scoring='f1_macro',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=1,
)
svm.fit(X_train, y_train)
predictions = svm.predict(X_test)
rows.append({
    'modelo': 'svm',
    'acuracia': accuracy_score(y_test, predictions),
    'macro_f1': f1_score(y_test, predictions, average='macro', zero_division=0),
    'balanced_accuracy': balanced_accuracy_score(y_test, predictions),
})
results = pd.DataFrame(rows).set_index('modelo').sort_values('macro_f1', ascending=False)
print('SVM selecionado no treino:', svm.best_params_)
results

SVM selecionado no treino: {'model__C': 10, 'model__kernel': 'rbf'}


,acuracia,macro_f1,balanced_accuracy
modelo,,,
svm,0.988,0.973475,0.979788
random_forest,0.986,0.968000,0.958644
regressao_logistica,0.946,0.878237,0.875717
naive_bayes,0.938,0.845357,0.811138
